In [33]:
import numpy as np
import pandas as pd

In [34]:
caption_tn_test = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/train_dataset/caption/tn_test.csv")
caption_tn_test["data_type"] = "caption_tn"

caption_tp_test = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/train_dataset/caption/tp_test.csv")
caption_tp_test["data_type"] = "caption_tp"


instruct_tn_test = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/train_dataset/instruct/tn_test.csv")
instruct_tn_test["data_type"] = "instruct_tn"

instruct_tp_test = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/train_dataset/instruct/tp_test.csv")
instruct_tp_test["data_type"] = "instruct_tp"

vqa_tn_test = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/train_dataset/vqa/tn_test.csv")
vqa_tn_test["data_type"] = "vqa_tn"

vqa_tp_test = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/train_dataset/vqa/tp_test.csv")
vqa_tp_test["data_type"] = "vqa_tp"

In [35]:
total_df = pd.concat([caption_tn_test, caption_tp_test, instruct_tn_test, instruct_tp_test, vqa_tn_test, vqa_tp_test])

In [36]:
total_df.index = range(len(total_df))

In [37]:
from tqdm import tqdm
all_target_words = []
failed = []
for inx, row in tqdm(total_df.iterrows()):
    try:
        res = eval(row["annotations"])
        obj_df = pd.DataFrame(res["object"])
        attr_df = pd.DataFrame(res["attribute"])
        rel_df = pd.DataFrame(res["relationship"])
        scene_df = pd.DataFrame(res["scene"])


        target_words = []
        if not obj_df.empty:
            for index, row in obj_df.iterrows():
                target_words.append({"word" : row["obj"]["name"], "type": "object", "label": "halu"})

        if not attr_df.empty:
            for index, row in attr_df.iterrows():
                target_words.append({"word" : row["attribute"]["name"], "type": "attribute", "label": "halu"})

        if not rel_df.empty:
        
            for index, row in rel_df.iterrows():
                target_words.append({"word" : row["predicate"]["name"], "type": "relationship", "label": "halu"})

        if not scene_df.empty:
            for index, row in scene_df.iterrows():
                target_words.append({"word" : row["scene"]["name"], "type": "scene", "label": "halu"})
        
        all_target_words.append(target_words)
    except:
        failed.append(inx)

26646it [00:22, 1204.23it/s]


In [38]:
len(failed)

1000

In [39]:
total_df = total_df.drop(index=failed)

In [40]:
total_df.shape

(25646, 12)

In [41]:
total_df["extracted_target_words"] = all_target_words

In [42]:
total_df.head(2)

,Unnamed: 0,image_id,prompt,hallucinated_text,source_text,source_metadata,qa_metadata,qa_ids,annotations,id,split,data_type,extracted_target_words
0,68069,2391368,<image>Can you describe the main features of t...,"In this image, there are some boxes contains f...","In this image, there are some boxes contains f...","{'source': 'localized_narratives', 'id': 'sp_2...",[],[],"{'object': [], 'attribute': [], 'relationship'...",caption_4_clean,test,caption_tn,[]
1,68070,2375207,<image>Write a detailed description of the giv...,A man is wearing a white uniform. He is wearin...,A man is wearing a white uniform. He is wearin...,"{'source': 'stanford', 'id': 'sp_30231'}",[],[],"{'object': [], 'attribute': [], 'relationship'...",caption_11_clean,test,caption_tn,[]


In [43]:
total_df["prompt"] = total_df["prompt"].apply(lambda x: x.replace("<image>", ""))

In [44]:
total_df.to_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/total_test_data_20k.csv", index=False)

In [87]:
target_test_data = total_df[total_df["data_type"].isin(["caption_tp"])]

In [88]:
target_test_data.shape

(3273, 13)

In [89]:
target_test_data.head(2)

,Unnamed: 0,image_id,prompt,hallucinated_text,source_text,source_metadata,qa_metadata,qa_ids,annotations,id,split,data_type,extracted_target_words
3273,64796,2391368,<image>Can you describe the main features of t...,"In this image, there are some boxes that conta...","In this image, there are some boxes contains f...","{'source': 'localized_narratives', 'id': 'sp_2...",[],['001006084'],"{'object': [], 'attribute': [{'attribute': {'n...",caption_4,test,caption_tp,"[{'word': 'wicker', 'type': 'attribute', 'labe..."
3274,64797,2375207,<image>Write a detailed description of the giv...,A man is wearing a white uniform. He is wearin...,A man is wearing a white uniform. He is wearin...,"{'source': 'stanford', 'id': 'sp_30231'}",[],['001021520'],"{'object': [], 'attribute': [{'attribute': {'n...",caption_11,test,caption_tp,"[{'word': 'plastic', 'type': 'attribute', 'lab..."


In [45]:
import pandas as pd
df = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/total_test_data_20k.csv")

In [46]:
target_df = df[["prompt", "hallucinated_text", "image_id", "data_type", "extracted_target_words", "id"]]

In [47]:
target_df.columns = ["question", "answer", "image_id", "data_type", "gt_answer", "question_id"]

In [54]:
target_df.shape

(25646, 7)

In [49]:
target_df["question"].iloc[-2]

'Use the provided image to answer the question: Which color is the camera? Provide your answer as short as possible:'

In [26]:
# from PIL import Image
# Image.open(target_df["image_path"].iloc[-2])

In [50]:
path = "/Data2/Arun-UAV/NLP/vision_halu/visual_genome/target_images/"

In [51]:
target_df["image_path"] = target_df["image_id"].apply(lambda x: path + str(x) + ".jpg")

/tmp/ipykernel_318029/3980793656.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_df["image_path"] = target_df["image_id"].apply(lambda x: path + str(x) + ".jpg")


In [52]:
target_df.to_csv("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/holoc/holoc_test_data.csv", index=False)

In [53]:
target_df["data_type"].value_counts()

data_type
vqa_tn         5041
instruct_tn    5009
instruct_tp    5009
vqa_tp         4041
caption_tp     3273
caption_tn     3273
Name: count, dtype: int64

In [57]:
import pandas as pd
result_df = pd.read_pickle("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/holoc/combined_stage_label_with_evidence_and_mlp_06_11_2025.pkl")

In [59]:
result_df.head(2)

,question,answer,question_id,image_id,image_path,gt_answer,data_type,labels_with_evidence
0,Can you describe the main features of this ima...,"In this image, there are some boxes contains f...",caption_4_clean,2391368,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,[],caption_tn,"[{'word': 'boxes', 'evidence': [0.02001849, 0...."
1,Write a detailed description of the given image.,A man is wearing a white uniform. He is wearin...,caption_11_clean,2375207,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,[],caption_tn,"[{'word': 'man', 'evidence': [0.04068803, 0.00..."


In [98]:
import torch
all_halu_words = []

for inx, row in tqdm(result_df.iterrows()):
    target_words = row["gt_answer"]
    
    pred_words = []
    for i in row["labels_with_evidence"]:
        viz_evi = (torch.tensor(i["evidence"]) >= 0.5).int().sum().item()
        prob = i["label"]
        if prob <= 0.6:
            pred_words.append(i["word"])
        elif viz_evi <= 6 and prob <= 0.9:
            pred_words.append(i["word"])
    
    all_halu_words.append(pred_words)

25646it [00:05, 4341.09it/s] 


In [99]:
result_df["predicted_tokens"] = all_halu_words

In [113]:

for name, target_test_data in result_df.groupby("data_type"):
    
    if "caption_tp" not in name:
        continue
    all_tp_words = []
    all_fn_words = []
    all_fp_words = []
    
    all_precision = []
    all_recall = []
    all_f1 = []
    failed = 0
    for inx, row in tqdm(target_test_data.iterrows()):
        gt_words =  [i["word"] for i in eval(row["gt_answer"])]
        pred_words = row["predicted_tokens"]
         
        tp_words = list(set(gt_words).intersection(set(pred_words)))
        all_tp_words.append(tp_words)
        fn_words = list(set(gt_words) - set(pred_words))
        all_fn_words.append(fn_words)
        fp_words = list(set(pred_words) - set(gt_words))
        all_fp_words.append(fp_words)
        if len(tp_words) == 0:
            failed += 1
            continue
        
        precision = len(tp_words)/(len(tp_words) + len(fp_words))
        all_precision.append(precision)
        recall = len(tp_words)/(len(tp_words) + len(fn_words))
        all_recall.append(recall)
        f1 = (2*precision*recall) / (precision + recall)
        all_f1.append(f1)

    # precision = len(all_tp_words)/(len(all_tp_words) + len(all_fp_words))
    # recall = len(all_tp_words)/(len(all_tp_words) + len(all_fn_words))
    # f1 = (2*precision*recall) / (precision + recall)

    # print(name, target_test_data.shape)
    # print(f"precision: {precision}")
    # print(f"recall: {recall}")
    # print(f"f1: {f1}")
    # print("\n----------------------\n")

0it [00:00, ?it/s]

3273it [00:00, 13221.77it/s]


In [114]:
sum(all_recall)/len(all_recall)

0.7587763185324167

In [115]:
sum(all_precision)/len(all_precision)

0.31497121828181945

In [116]:
sum(all_f1)/len(all_f1)

0.40615259806032683